
# 07 — Planner, Writer and Critic: Iterative AI Workflow

**Learning reference:** [BrightPool — Prompt Engineering Course](https://github.com/BrightPool/udemy-prompt-engineering-course)

**Inspired by course notebooks:**
- `blog_post_generator_agent.ipynb`
- `introduction_to_openai_ai_agents_sdk.ipynb`

## Goal

Understand and implement a Planner → Writer → Critic
workflow for generating and reviewing an article.

This notebook contains two implementations:

1. **Offline workflow:** Deterministic Python functions
   demonstrate orchestration, feedback, revision, and stopping.
2. **Live AI workflow:** Gemini performs planning, writing,
   criticism, and an optional revision.

The implementations are inspired by the course's
multi-agent architecture but use Python orchestration
rather than the OpenAI Agents SDK.

**API usage:** The offline section requires zero requests.
The live section requires three Gemini requests, plus
one optional revision request.

The article topic is educational and uses no private data.



## 1. Understanding the Roles

**Planner**

Transforms a topic into an outline with clear sections.

**Writer**

Produces an article based on the plan and improves
the draft when feedback is provided.

**Critic**

Evaluates whether the draft meets its requirements
and identifies areas for improvement.

**Orchestrator**

Controls the sequence of steps and decides whether
another revision is necessary.

The roles do not have to use different AI models.
The same model can perform different roles by receiving
different instructions.

We first implement the workflow without AI to understand
its control flow and stopping conditions.


In [4]:

from dataclasses import dataclass

MAX_ITERATIONS = 3
QUALITY_THRESHOLD = 3


@dataclass
class Critique:
    score: int
    missing: list[str]
    passed: bool


def plan_article(topic):
    return {
        "title": topic,
        "sections": [
            "Introduction",
            "Example",
            "Limitations"
        ]
    }


def write_article(plan, feedback=None):
    sections = {
        "Introduction": (
            f"{plan['title']} explains a useful "
            "engineering idea."
        ),
        "Example": (
            "Example: a bounded workflow stops after "
            "a fixed number of iterations."
        ),
        "Limitations": (
            "Limitations: human review remains "
            "necessary before publication."
        )
    }

    # Deliberately omit a section from the first draft.
    if feedback is None:
        sections.pop("Limitations")

    return "\n\n".join(
        f"## {section}\n{body}"
        for section, body in sections.items()
    )


def critique_article(draft, plan):
    missing = [
        section
        for section in plan["sections"]
        if f"## {section}\n" not in draft
    ]

    score = len(plan["sections"]) - len(missing)

    return Critique(
        score=score,
        missing=missing,
        passed=score >= QUALITY_THRESHOLD
    )


print("Offline Planner, Writer and Critic ready.")


Offline Planner, Writer and Critic ready.



## 2. Bounded Orchestration

A revision workflow needs a stopping condition.

Without a limit, the Writer and Critic could continue
revising indefinitely.

Our offline orchestrator stops when either:

- The draft meets the required quality threshold.
- The maximum number of iterations is reached.

In this simplified example, quality means that all
required headings are present. It does not measure
the article's factual accuracy or writing quality.


In [5]:

def generate_article(topic, max_iterations=MAX_ITERATIONS):
    if max_iterations < 1:
        raise ValueError(
            "max_iterations must be at least 1"
        )

    plan = plan_article(topic)
    feedback = None
    history = []

    for iteration in range(1, max_iterations + 1):
        draft = write_article(plan, feedback)
        critique = critique_article(draft, plan)

        history.append({
            "iteration": iteration,
            "score": critique.score,
            "missing": critique.missing
        })

        if critique.passed:
            return {
                "draft": draft,
                "passed": True,
                "iterations": iteration,
                "history": history
            }

        feedback = critique.missing

    return {
        "draft": draft,
        "passed": False,
        "iterations": max_iterations,
        "history": history
    }


offline_result = generate_article("Reliable AI Agents")

print("Passed:", offline_result["passed"])
print("Iterations:", offline_result["iterations"])
print("Critique history:", offline_result["history"])

print("\nFINAL ARTICLE:\n")
print(offline_result["draft"])


Passed: True
Iterations: 2
Critique history: [{'iteration': 1, 'score': 2, 'missing': ['Limitations']}, {'iteration': 2, 'score': 3, 'missing': []}]

FINAL ARTICLE:

## Introduction
Reliable AI Agents explains a useful engineering idea.

## Example
Example: a bounded workflow stops after a fixed number of iterations.

## Limitations
Limitations: human review remains necessary before publication.



## 3. Offline Verification

Test that the orchestrator:

- Revises an incomplete draft.
- Stops once the required threshold is met.
- Respects the maximum iteration limit.

These tests execute locally without calling Gemini.


In [6]:

assert offline_result["passed"] is True
assert offline_result["iterations"] == 2

assert [
    item["score"]
    for item in offline_result["history"]
] == [2, 3]

assert "## Limitations" in offline_result["draft"]


limit_test = generate_article(
    "Reliable AI Agents",
    max_iterations=1
)

assert limit_test["passed"] is False
assert limit_test["iterations"] == 1

print("All offline workflow tests passed.")
print("Gemini API requests: 0")


All offline workflow tests passed.
Gemini API requests: 0



## 4. Offline Findings

The deterministic implementation demonstrates:

- Separating planning, drafting and evaluation.
- Passing feedback between roles.
- Revising a draft based on missing requirements.
- Early stopping and maximum iteration limits.

**Limitation:** The Writer uses predefined text and
the Critic checks headings only.

The next section replaces these deterministic roles
with live Gemini calls to produce a real article,
evaluate its content, and optionally revise it.



## 5. Live AI Workflow with Gemini

Use Gemini as three independent roles:

1. **AI Planner:** Generates a structured outline.
2. **AI Writer:** Writes an original article from the outline.
3. **AI Critic:** Reviews completeness, clarity and relevance.
4. **AI Revision:** Improves the draft if the Critic requests changes.

Each role uses a separate prompt and API call.

The workflow uses Python to coordinate these roles.
It is a role-based LLM workflow, not an implementation
of the OpenAI Agents SDK.

To control API usage:

- Automatic retries are disabled.
- Each live step has its own Colab cell.
- The revision request is optional.
- A previous successful step does not need to run again
  when a later step fails.


In [7]:

from openai import OpenAI
from getpass import getpass
import json

MODEL = "gemini-3.5-flash-lite"

gemini_key = getpass("Enter your Gemini API key: ")

client = OpenAI(
    api_key=gemini_key,
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
    max_retries=0,
    timeout=120.0
)

safe_client = client.with_options(
    max_retries=0,
    timeout=120.0
)

TOPIC = "Reliable AI Agents"

print("Model:", MODEL)
print("Topic:", TOPIC)
print("Client ready. No API request made.")


Enter your Gemini API key: ··········
Model: gemini-3.5-flash-lite
Topic: Reliable AI Agents
Client ready. No API request made.



## 6. AI Planner

The Planner receives the topic and produces an outline.

We ask for the following structure:

- Introduction
- How It Works
- Practical Example
- Limitations
- Conclusion

The Planner should describe what each section will cover.

This is the first live Gemini request.


In [8]:

planner_response = safe_client.chat.completions.create(
    model=MODEL,
    messages=[
        {
            "role": "system",
            "content": (
                "You are an editorial planner. "
                "Create clear, practical article outlines. "
                "Focus on technical accuracy and readability."
            )
        },
        {
            "role": "user",
            "content": (
                f"Create an outline for an article titled '{TOPIC}'. "
                "The audience is beginner-to-intermediate Python "
                "developers learning about AI agents. "
                "Use these five sections in this exact order: "
                "Introduction, How It Works, Practical Example, "
                "Limitations, Conclusion. "
                "For each section, include 2-3 points to cover. "
                "Return plain Markdown."
            )
        }
    ],
    temperature=0.3
)

ai_plan = planner_response.choices[0].message.content

if not ai_plan:
    raise ValueError("Planner returned empty content.")

print("AI PLANNER OUTPUT:\n")
print(ai_plan)


AI PLANNER OUTPUT:

# Reliable AI Agents

## Introduction
- Define what an AI agent is beyond a standard LLM prompt (incorporating perception, reasoning, and tool use/action).
- Explain why "reliability" is the primary challenge in production systems (handling hallucination, infinite loops, and unexpected tool outputs).
- Set expectations for what readers will learn about building robust, deterministic agent workflows in Python.

## How It Works
- Break down the core architectural components: the LLM (brain), the Memory (state), and the Tools (hands/actions).
- Explain the core reasoning loops (such as ReAct: Reason, Act, Observe) and how they guide step-by-step execution.
- Highlight the mechanisms used to enforce reliability, including structured output parsing, guardrails, and deterministic fallback logic.

## Practical Example
- Introduce a simple Python implementation using a popular framework (like LangChain or LlamaIndex) or native function calling to build a reliable web-search

In [9]:

writer_response = safe_client.chat.completions.create(
    model=MODEL,
    messages=[
        {
            "role": "system",
            "content": (
                "You are a technical writer. "
                "Write original, clear and useful educational "
                "articles for Python developers. "
                "Avoid unsupported claims and unnecessary jargon."
            )
        },
        {
            "role": "user",
            "content": (
                f"Write an article about '{TOPIC}' "
                "using the following outline:\n\n"
                f"{ai_plan}\n\n"
                "Requirements:\n"
                "- Approximately 350-500 words.\n"
                "- Use these exact Markdown headings: "
                "## Introduction, ## How It Works, "
                "## Practical Example, ## Limitations, "
                "## Conclusion.\n"
                "- Include a concrete example of an agent "
                "using a tool.\n"
                "- Explain at least one practical limitation.\n"
                "- Keep the language accessible.\n"
                "- Return only the article in Markdown."
            )
        }
    ],
    temperature=0.5
)

ai_draft = writer_response.choices[0].message.content

if not ai_draft:
    raise ValueError("Writer returned empty content.")

print("AI WRITER — FIRST DRAFT:\n")
print(ai_draft)


AI WRITER — FIRST DRAFT:

# Reliable AI Agents

## Introduction
An AI agent goes beyond a standard Large Language Model (LLM) prompt by combining perception, reasoning, and tool use to achieve multi-step goals autonomously. While chatbots simply respond to text, agents plan actions, query APIs, and modify their behavior based on results. 

However, moving from a clever demo to production reveals the primary challenge: reliability. Handling hallucinations, infinite execution loops, and unexpected API responses requires moving past basic prompt engineering. This article explores how to build robust, deterministic agent workflows in Python.

## How It Works
Reliable agents rely on three architectural components:
1. **The LLM (Brain):** Generates plans and interprets data.
2. **The Memory (State):** Tracks the conversation history and intermediate steps.
3. **The Tools (Hands):** Functions the agent can execute, such as database queries or web searches.

Most agents use the ReAct (Reason, 


## 7. AI Critic

The Critic evaluates the first draft using these criteria:

- All five required sections are present.
- The explanation is understandable to the intended audience.
- A concrete tool-using agent example is included.
- Relevant limitations are discussed.
- The article stays focused on the requested topic.

The Critic provides:

- A score from 1 to 5.
- A pass/fail decision.
- Specific feedback for improvement.

The feedback is returned as JSON so Python can
use it in the revision step.

The score is the model's subjective assessment.
It is not an independently verified quality metric.


In [10]:

critic_response = safe_client.chat.completions.create(
    model=MODEL,
    messages=[
        {
            "role": "system",
            "content": (
                "You are a critical technical editor. "
                "Evaluate drafts consistently and constructively. "
                "Do not rewrite the article. "
                "Return a valid JSON object and no other text."
            )
        },
        {
            "role": "user",
            "content": (
                "Review the article below.\n\n"
                "Requirements:\n"
                "1. Five required Markdown sections.\n"
                "2. Clear explanation for Python learners.\n"
                "3. Concrete agent tool-use example.\n"
                "4. Meaningful limitations.\n"
                "5. Relevance to the topic.\n\n"
                "Return JSON with exactly these fields:\n"
                '{ "score": 1, "passed": false, '
                '"feedback": ["Specific suggestion"] }\n\n'
                "Use an integer score from 1 to 5.\n"
                "Set passed=true only for scores of 4 or 5 "
                "when all requirements are satisfied.\n"
                "If passed=true, feedback may be an empty list.\n\n"
                f"ARTICLE:\n{ai_draft}"
            )
        }
    ],
    temperature=0
)

critic_text = critic_response.choices[0].message.content

if not critic_text:
    raise ValueError("Critic returned empty content.")

# Handle an optional Markdown JSON code fence.
clean_text = critic_text.strip()

if clean_text.startswith("```"):
    clean_text = clean_text.split("\n", 1)[1]
    clean_text = clean_text.rsplit("```", 1)[0].strip()

ai_critique = json.loads(clean_text)

if not isinstance(ai_critique, dict):
    raise ValueError("Critic output must be a JSON object.")

if not (
    isinstance(ai_critique.get("score"), int)
    and 1 <= ai_critique["score"] <= 5
    and isinstance(ai_critique.get("passed"), bool)
    and isinstance(ai_critique.get("feedback"), list)
    and all(isinstance(x, str) for x in ai_critique["feedback"])
):
    raise ValueError("Critic returned an invalid evaluation schema.")

print("CRITIC SCORE:", ai_critique["score"])
print("PASSED:", ai_critique["passed"])
print("FEEDBACK:")

for item in ai_critique["feedback"]:
    print("-", item)


CRITIC SCORE: 5
PASSED: True
FEEDBACK:


In [11]:

final_article = ai_draft
revision_performed = False

if ai_critique["passed"]:
    print("Critic approved the first draft.")
    print("No revision request needed.")

else:
    print("Critic requested improvements.")
    print("Sending one revision request...")

    revision_response = safe_client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "system",
                "content": (
                    "You are a technical writer revising "
                    "an article based on editorial feedback. "
                    "Preserve correct information and improve "
                    "weak or incomplete sections."
                )
            },
            {
                "role": "user",
                "content": (
                    f"TOPIC:\n{TOPIC}\n\n"
                    f"ORIGINAL PLAN:\n{ai_plan}\n\n"
                    f"CURRENT DRAFT:\n{ai_draft}\n\n"
                    "CRITIC FEEDBACK:\n"
                    f"{json.dumps(ai_critique['feedback'])}\n\n"
                    "Revise the article to address the feedback. "
                    "Use the five required Markdown headings. "
                    "Return only the revised article."
                )
            }
        ],
        temperature=0.3
    )

    revised_text = revision_response.choices[0].message.content

    if not revised_text:
        raise ValueError("Revision returned empty content.")

    final_article = revised_text
    revision_performed = True

    print("Revision completed.")

print("\nFINAL ARTICLE:\n")
print(final_article)


Critic approved the first draft.
No revision request needed.

FINAL ARTICLE:

# Reliable AI Agents

## Introduction
An AI agent goes beyond a standard Large Language Model (LLM) prompt by combining perception, reasoning, and tool use to achieve multi-step goals autonomously. While chatbots simply respond to text, agents plan actions, query APIs, and modify their behavior based on results. 

However, moving from a clever demo to production reveals the primary challenge: reliability. Handling hallucinations, infinite execution loops, and unexpected API responses requires moving past basic prompt engineering. This article explores how to build robust, deterministic agent workflows in Python.

## How It Works
Reliable agents rely on three architectural components:
1. **The LLM (Brain):** Generates plans and interprets data.
2. **The Memory (State):** Tracks the conversation history and intermediate steps.
3. **The Tools (Hands):** Functions the agent can execute, such as database queries o

In [12]:

required_headings = [
    "## Introduction",
    "## How It Works",
    "## Practical Example",
    "## Limitations",
    "## Conclusion"
]

missing_headings = [
    heading
    for heading in required_headings
    if heading not in final_article
]

print("MODEL:", MODEL)
print("TOPIC:", TOPIC)

print("\nCritic score on initial draft:",
      ai_critique["score"])

print("Initial draft approved:",
      ai_critique["passed"])

print("Revision performed:", revision_performed)
print("Missing required headings:", missing_headings)

print("\nWord count:", len(final_article.split()))

if missing_headings:
    print("\nStructural verification: FAILED")
    print("Review the final article before publishing.")
else:
    print("\nStructural verification: PASSED")

if revision_performed:
    print(
        "Note: The revised article has not been "
        "re-evaluated by the AI Critic."
    )


MODEL: gemini-3.5-flash-lite
TOPIC: Reliable AI Agents

Critic score on initial draft: 5
Initial draft approved: True
Revision performed: False
Missing required headings: []

Word count: 500

Structural verification: PASSED



## Findings and Limitations

### Offline Workflow

The deterministic workflow demonstrated:
- Separating planning, writing and criticism.
- Using feedback to trigger revisions.
- Early stopping when requirements are satisfied.
- Enforcing a maximum number of iterations.

### Live Gemini Workflow

The live workflow uses separate LLM calls for:
- Creating an article outline.
- Writing an article from that outline.
- Evaluating the initial draft against explicit criteria.
- Optionally revising the article using feedback.

Python coordinates the roles and stores the outputs
between steps.

### Limitations

- The roles share one model rather than using separate models.
- This implementation does not use the OpenAI Agents SDK.
- The AI Critic's score is subjective.
- Factual claims are not externally verified.
- At most one live revision is performed.
- Revised drafts are not re-evaluated by the AI Critic.
- API timeouts and service errors may interrupt execution.

### Conclusion

The exercise illustrates how role separation, structured
feedback and bounded execution can support more
controlled AI content-generation workflows.

Actual success and observed Critic feedback should be
assessed from the saved notebook outputs.
